# Stage 4 — Champion Validation, Selection-Criterion Repair & Uniformity Anti-Collapse

**Brief:** `briefs/BRIEF-007-stage-4-brief.md`

**Objectives:**
1. **$\lambda_{	ext{spa}}$ Calibration:** Empirically calibrate $\lambda_{	ext{spa}}$ on E15 (seed 42) into the reconstruction-alive band $R^2_{	ext{rna}} \in [0.10, 0.35]$.
2. **3A. Champion Validation:** Validate the evidence-based champion (`S4-CHAMP-gat`: GAT + HierMLP + PCA30 recon + calibrated $\lambda_{	ext{spa}}^*$ + IDEC) and its SAGE backbone ablation (`S4-CHAMP-sage`).
3. **Calibrated Baseline:** `S4-BASE-recal` for direct head-to-head comparison.
4. **3C. Uniformity Anti-Collapse:** Wang & Isola (2020) uniformity regularization on the calibrated base (`S4-ABL-unif`).
5. **3B. Selection-Criterion Repair:** Apply RAUS v3b ($	ext{Borda}\{	ext{Stability}, 	ext{RecBal}, R^2_{	ext{rna}}\}$) with verified Spearman $ho pprox 0.62$.


In [ ]:
import os, sys, json, subprocess, random, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for pkg in ["scanpy", "anndata", "scikit-learn", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import scanpy as sc
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans
from sklearn.neighbors import kneighbors_graph
from scipy import sparse
import pandas as pd

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

print("imports ok")


In [ ]:
# ---------- GitHub push (guarded; clone-or-pull fixes stale-dir failure) ----------
REPO = "/kaggle/working/my_research"
REPO_URL = "https://github.com/rifatahsanpul0k/my_research.git"
def sh(*args):
    subprocess.run(list(args), check=True, cwd=REPO)
os.makedirs(REPO, exist_ok=True)
OUT = os.path.join(REPO, "runs", "s4")
os.makedirs(OUT, exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    import base64, shutil
    _sec = UserSecretsClient()
    _tok = None
    try:
        _tok = _sec.get_secret('GITHUB_TOKEN')
    except Exception:
        pass
    if not _tok:
        _tok = None
    if _tok:
        _b64 = base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
        _GIT = ["git", "-c", f"http.extraHeader=Authorization: Basic {_b64}"]
        if not os.path.isdir(os.path.join(REPO, ".git")):
            if os.listdir(REPO):
                bak = REPO + "_stalebak"
                if os.path.exists(bak):
                    shutil.rmtree(bak)
                shutil.move(REPO, bak)
                os.makedirs(REPO, exist_ok=True)
            subprocess.run(_GIT + ["clone", REPO_URL, REPO], check=True)
            sh("git", "config", "user.name", "kaggle-runner")
            sh("git", "config", "user.email", "kaggle-runner@local")
        else:
            sh(*_GIT, "pull", "--rebase", "origin", "main")
        GIT_OK = True
        print("GitHub sync ready")
    else:
        _GIT = ["git"]
        GIT_OK = False
        print("GitHub sync unavailable (local save only)")
except Exception as e:
    GIT_OK = False
    print("GitHub sync unavailable (local save only):", e)

def git_push(paths, msg):
    if not GIT_OK:
        return
    try:
        sh(*_GIT, "pull", "--rebase", "origin", "main")
        sh("git", "add", *paths)
        st = subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True)
        if not st.stdout.strip():
            return
        sh("git", "commit", "-m", msg)
        sh(*_GIT, "push", "origin", "main")
        print("pushed:", msg)
    except Exception as e:
        print("push failed (continuing):", e)


In [ ]:
# ---------- data loading & cached preprocessing ----------
MARKER = "10x_human_lymph_node_A1"
def find_data_root():
    for base in ["/kaggle/input/spatial-multiomics-6datasets-private",
                 "/kaggle/working/spatial-multiomics-6datasets-private",
                 "/kaggle/working"]:
        if os.path.isdir(os.path.join(base, MARKER)):
            return base
    return None

DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        kd = os.path.expanduser("~/.kaggle"); os.makedirs(kd, exist_ok=True)
        json.dump({"username": _s.get_secret("KAGGLE_USERNAME"), "key": _s.get_secret("KAGGLE_KEY")},
                  open(os.path.join(kd, "kaggle.json"), "w"))
        os.chmod(os.path.join(kd, "kaggle.json"), 0o600)
        subprocess.run(["kaggle", "datasets", "download", "-d", "pulokpulok/spatial-multiomics-6datasets-private",
                        "-p", "/kaggle/working", "--unzip"], check=True)
        DATA_ROOT = find_data_root()
    except Exception as e:
        print("Dataset auto-download failed:", e)

if DATA_ROOT is None:
    raise FileNotFoundError("dataset folders not found: " + str(sorted(os.listdir("/kaggle/working"))))
print("DATA_ROOT:", DATA_ROOT)

DATASETS = {
    "A1":  dict(folder="10x_human_lymph_node_A1", aux="adt",  gt_file="annotation.csv", gt_col="manual-anno"),
    "D1":  dict(folder="10x_human_lymph_node_D1", aux="adt",  gt_file="annotation.csv", gt_col="manual-anno"),
    "E11": dict(folder="Mouse_Brain_E11_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E13": dict(folder="Mouse_Brain_E13_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E15": dict(folder="Mouse_Brain_E15_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E18": dict(folder="Mouse_Brain_E18_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
}
DECLARED_K = {"A1": 10, "D1": 11, "E11": 8, "E13": 12, "E15": 12, "E18": 14}
SEEDS = [42, 1234, 2024]

def to_dense_small(X, max_elems=5e7):
    if hasattr(X, "toarray"):
        if X.shape[0] * X.shape[1] <= max_elems:
            return X.toarray()
        return X.tocsr()
    return np.asarray(X)

def load_dataset(ds_id):
    meta = DATASETS[ds_id]
    p = os.path.join(DATA_ROOT, meta["folder"])
    adata_rna = sc.read_h5ad(os.path.join(p, "adata_RNA.h5ad"))
    coords = np.asarray(adata_rna.obsm["spatial"], dtype=np.float32)

    Xr_raw = to_dense_small(adata_rna.X)
    if sparse.issparse(Xr_raw):
        Xr_raw = Xr_raw.toarray()
    adata_rna.X = Xr_raw.copy()
    sc.pp.normalize_total(adata_rna, target_sum=1e4)
    sc.pp.log1p(adata_rna)
    try:
        sc.pp.highly_variable_genes(adata_rna, n_top_genes=min(3000, adata_rna.n_vars), flavor="seurat_v3")
    except Exception:
        sc.pp.highly_variable_genes(adata_rna, n_top_genes=min(3000, adata_rna.n_vars), flavor="seurat")
    hvg_idx = adata_rna.var["highly_variable"].values
    Xh = Xr_raw[:, hvg_idx] if hvg_idx.sum() >= 100 else Xr_raw[:, :min(3000, Xr_raw.shape[1])]

    sc.pp.scale(adata_rna, max_value=10)
    Xr_pca64 = PCA(n_components=64, random_state=42).fit_transform(adata_rna.X)
    Xr_pca30 = Xr_pca64[:, :30]

    aux_type = meta["aux"]
    if aux_type == "adt":
        adata_aux = sc.read_h5ad(os.path.join(p, "adata_ADT.h5ad"))
        Xa_raw = to_dense_small(adata_aux.X)
        if sparse.issparse(Xa_raw):
            Xa_raw = Xa_raw.toarray()
        adata_aux.X = Xa_raw.copy()
        sc.pp.normalize_total(adata_aux, target_sum=1e4)
        sc.pp.log1p(adata_aux)
        sc.pp.scale(adata_aux, max_value=10)
        n_comp = min(64, adata_aux.shape[1] - 1)
        Xa = PCA(n_components=n_comp, random_state=42).fit_transform(adata_aux.X)
    else:
        adata_aux = sc.read_h5ad(os.path.join(p, "adata_ATAC.h5ad"))
        Xa_raw = to_dense_small(adata_aux.X)
        from sklearn.decomposition import TruncatedSVD
        Xa = TruncatedSVD(n_components=64, random_state=42).fit_transform(Xa_raw)
        Xa = StandardScaler().fit_transform(Xa)

    gt_df = pd.read_csv(os.path.join(p, meta["gt_file"]))
    gt = pd.Categorical(gt_df[meta["gt_col"]]).codes
    return {
        "Xr_pca64": Xr_pca64.astype(np.float32),
        "Xr_pca30": Xr_pca30.astype(np.float32),
        "Xh": Xh.astype(np.float32),
        "Xa": Xa.astype(np.float32),
        "coords": coords,
        "gt": gt,
        "k": DECLARED_K[ds_id]
    }


In [ ]:
# ---------- graph & spatial helpers ----------
def knn_edge_index(coords, k=6):
    nbrs = kneighbors_graph(coords, n_neighbors=k, mode="connectivity", include_self=False)
    coo = nbrs.tocoo()
    return torch.tensor(np.stack([coo.row, coo.col]), dtype=torch.long)

def norm_adj_nl(edge_index, n, device):
    src, dst = edge_index[0], edge_index[1]
    deg = torch.zeros(n, device=device).scatter_add_(0, dst, torch.ones_like(src, dtype=torch.float32))
    deg_inv = torch.where(deg > 0, deg.pow(-0.5), torch.zeros_like(deg))
    w = deg_inv[src] * deg_inv[dst]
    return torch.sparse_coo_tensor(torch.stack([dst, src]), w, (n, n), device=device)

def build_graph(coords, device):
    n = coords.shape[0]
    ei = knn_edge_index(coords, 6).to(device)
    return dict(edge_index=ei, adj_nl=norm_adj_nl(ei, n, device))


In [ ]:
# ---------- neural network modules ----------
D_EMB, D_HID = 32, 128

class SAGEEncoder(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.l1 = nn.Linear(2 * d_in, D_HID); self.l2 = nn.Linear(2 * D_HID, D_EMB)
    def forward(self, x, g):
        agg = torch.sparse.mm(g["adj_nl"], x)
        h = self.l1(torch.cat([x, agg], 1)).relu()
        agg2 = torch.sparse.mm(g["adj_nl"], h)
        return self.l2(torch.cat([h, agg2], 1))

class GATEncoder(nn.Module):  # Velickovic et al. 2018 LeakyReLU fidelity
    def __init__(self, d_in, heads=4, dh=32):
        super().__init__()
        self.heads, self.dh = heads, dh
        self.W = nn.Linear(d_in, dh * heads, bias=False)
        self.a = nn.Parameter(torch.empty(heads, 2 * dh))
        nn.init.xavier_uniform_(self.a)
        self.out = nn.Linear(dh * heads, D_EMB)
    def forward(self, x, g):
        ei = g["edge_index"]; n = x.size(0); H, dh = self.heads, self.dh
        h = self.W(x).view(n, H, dh)
        src, dst = ei[0], ei[1]
        e = torch.cat([h[src], h[dst]], -1)
        al = torch.exp(F.leaky_relu((e * self.a).sum(-1), 0.2))
        den = torch.zeros(n, H, device=x.device).scatter_add_(0, dst.unsqueeze(-1).expand(-1, H), al)
        al = al / (den[dst] + 1e-9)
        o = torch.zeros(n, H, dh, device=x.device).scatter_add_(
            0, dst.view(-1, 1, 1).expand(-1, H, dh), al.unsqueeze(-1) * h[src])
        return self.out(o.reshape(n, H * dh).relu())

ENCODERS = {"sage": SAGEEncoder, "gat": GATEncoder}

class MLPHierFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.s1 = nn.Sequential(nn.Linear(2 * D_EMB, D_EMB), nn.ReLU())
        self.s2 = nn.Sequential(nn.Linear(2 * D_EMB, D_EMB), nn.ReLU(), nn.Linear(D_EMB, D_EMB))
    def forward(self, zr, za):
        h1 = self.s1(torch.cat([zr, za], 1))
        return self.s2(torch.cat([h1, zr + za], 1))

FUSIONS = {"mlphier": MLPHierFusion}


In [ ]:
# ---------- loss functions (Laplacian, Uniformity, R2, IDEC) ----------
def laplacian_loss(z, edge_index):
    src, dst = edge_index
    return 0.5 * ((z[src] - z[dst]).pow(2).sum(1)).mean()

def uniformity_loss(z, t=2.0, max_samples=1024):
    """Wang & Isola (2020) uniformity loss on the hypersphere."""
    n = z.size(0)
    if n > max_samples:
        idx = torch.randperm(n, device=z.device)[:max_samples]
        z_sub = z[idx]
    else:
        z_sub = z
    z_norm = F.normalize(z_sub, dim=-1)
    sim = torch.mm(z_norm, z_norm.t())
    dist_sq = 2.0 - 2.0 * sim
    m = z_sub.size(0)
    mask = ~torch.eye(m, dtype=torch.bool, device=z.device)
    val = -t * dist_sq[mask]
    return torch.logsumexp(val, dim=0) - math.log(val.numel())

def r2_score(pred, true):
    ss_res = ((true - pred) ** 2).sum()
    ss_tot = ((true - true.mean(0)) ** 2).sum()
    return torch.clamp(1 - ss_res / (ss_tot + 1e-9), 0.0, 1.0)

def recon_balance(r2_rna, r2_aux):
    return float((min(r2_rna, r2_aux) + 1e-4) / (max(r2_rna, r2_aux) + 1e-4))

def student_q(z, mu, alpha=1.0):
    d2 = torch.cdist(z, mu).pow(2)
    q = (1 + d2 / alpha).pow(-(alpha + 1) / 2)
    return q / q.sum(1, keepdim=True)

def target_p(q):
    f = q.sum(0, keepdim=True)
    p = (q.pow(2) / f)
    return p / p.sum(1, keepdim=True)


In [ ]:
# ---------- Stage 4 Model ----------
class S4Model(nn.Module):
    def __init__(self, d_rna, d_aux, cfg, d_recon_rna):
        super().__init__()
        self.cfg = cfg
        self.enc_r = ENCODERS[cfg["enc"]](d_rna)
        self.enc_a = ENCODERS[cfg["enc"]](d_aux)
        self.fusion = FUSIONS[cfg["fusion"]]()
        self.dec_r = nn.Linear(D_EMB, d_recon_rna)
        self.dec_a = nn.Linear(D_EMB, d_aux)

    def forward(self, xr, xa, g):
        zr = self.enc_r(xr, g); za = self.enc_a(xa, g)
        z = self.fusion(zr, za)
        return z, zr, za

    def loss(self, z, zr, za, tr, ta, g):
        L = F.mse_loss(self.dec_r(z), tr) + F.mse_loss(self.dec_a(z), ta)
        lam_spa = self.cfg.get("lam_spa", 0.1)
        if lam_spa > 0:
            L = L + lam_spa * laplacian_loss(z, g["edge_index"])
        if self.cfg.get("unif", False):
            L = L + 0.1 * uniformity_loss(z)
        return L


In [ ]:
# ---------- training & execution helpers ----------
def recon_target(cfg, data):
    tgt = cfg.get("recon_target", "pca64")
    if tgt == "pca30":
        return data["Xr_pca30"], 30
    elif tgt == "hvg3000":
        return data["Xh"], data["Xh"].shape[1]
    return data["Xr_pca64"], 64

def train_embed(cfg, data, g, seed, epochs=120, lr=1e-3, kl_head=False):
    set_seed(seed)
    xr = torch.tensor(data["Xr_pca64"], device=DEVICE)
    xa = torch.tensor(data["Xa"], device=DEVICE)
    tr_np, d_rr = recon_target(cfg, data)
    tr = torch.tensor(tr_np, device=DEVICE)
    ta = xa
    model = S4Model(data["Xr_pca64"].shape[1], data["Xa"].shape[1], cfg, d_rr).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    mu = None
    k = data["k"]
    model.train()
    for ep in range(epochs):
        opt.zero_grad()
        z, zr, za = model(xr, xa, g)
        loss = model.loss(z, zr, za, tr, ta, g)
        if kl_head:  # IDEC: joint KL + recon
            if ep % 25 == 0 or mu is None:
                with torch.no_grad():
                    ze = z.detach().cpu().numpy()
                    km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(ze)
                    mu = torch.tensor(km.cluster_centers_, dtype=torch.float32, device=DEVICE)
            q = student_q(z, mu); p = target_p(q).detach()
            loss = loss + 0.1 * F.kl_div(q.log(), p, reduction="batchmean")
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        z, zr, za = model(xr, xa, g)
        r2r = float(r2_score(model.dec_r(z), tr)); r2a = float(r2_score(model.dec_a(z), ta))
    return z.detach().cpu().numpy(), r2r, r2a, (mu.detach() if mu is not None else None)


In [ ]:
# ---------- Protocol 4: Lambda Calibration on E15 (seed 42) ----------
print("Preprocessing datasets...", flush=True)
DATA_CACHE = {}
for ds_id in DATASETS:
    d = load_dataset(ds_id); d["_ds"] = ds_id; DATA_CACHE[ds_id] = d
print("Data preprocessed & cached.")

g_e15 = build_graph(DATA_CACHE["E15"]["coords"], DEVICE)
candidates_lam = [2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01]
calib_results = []
print("\n=== Lambda Calibration on E15, seed 42 (Base config) ===")
cfg_calib = dict(enc="sage", fusion="mlphier", spatial="laplacian", recon_target="pca64", head="kmeans")

for lam in candidates_lam:
    cfg_test = dict(cfg_calib, lam_spa=lam)
    Z_c, r2r, r2a, _ = train_embed(cfg_test, DATA_CACHE["E15"], g_e15, seed=42)
    in_band = (0.10 <= r2r <= 0.35)
    print(f"lambda_spa = {lam:4.2f} -> R2_rna = {r2r:.4f}, R2_aux = {r2a:.4f} [in_band={in_band}]", flush=True)
    calib_results.append({"lam_spa": lam, "r2_rna": r2r, "r2_aux": r2a, "in_band": in_band})

in_band_cands = [r for r in calib_results if r["in_band"]]
if in_band_cands:
    # Select candidate closest to target center 0.225
    CALIB_LAMBDA = min(in_band_cands, key=lambda r: abs(r["r2_rna"] - 0.225))["lam_spa"]
    print(f"\n--> CALIBRATED LAMBDA_SPA: {CALIB_LAMBDA} (reconstruction-alive band achieved)")
else:
    CALIB_LAMBDA = min(calib_results, key=lambda r: abs(r["r2_rna"] - 0.20))["lam_spa"]
    print(f"\n--> Closest lambda_spa selected: {CALIB_LAMBDA}")

json.dump({"calibrated_lambda": CALIB_LAMBDA, "trajectory": calib_results},
          open(os.path.join(OUT, "calibration_e15.json"), "w"), indent=2)
git_push(["runs/s4/calibration_e15.json"], f"s4: calibrated lambda_spa = {CALIB_LAMBDA}")


In [ ]:
# ---------- Stage 4 Matrix Definition ----------
COMBINATIONS = [
    # 3A. Champion validation
    ("S4-CHAMP-gat",  dict(enc="gat",  fusion="mlphier", spatial="laplacian", lam_spa=CALIB_LAMBDA, recon_target="pca30", head="idec")),
    ("S4-CHAMP-sage", dict(enc="sage", fusion="mlphier", spatial="laplacian", lam_spa=CALIB_LAMBDA, recon_target="pca30", head="idec")),
    # Calibrated Base reference
    ("S4-BASE-recal", dict(enc="sage", fusion="mlphier", spatial="laplacian", lam_spa=CALIB_LAMBDA, recon_target="pca64", head="kmeans")),
    # 3C. First new idea: Uniformity anti-collapse
    ("S4-ABL-unif",   dict(enc="sage", fusion="mlphier", spatial="laplacian", lam_spa=CALIB_LAMBDA, recon_target="pca64", head="kmeans", unif=True)),
]
print(f"Stage 4 registered {len(COMBINATIONS)} configurations with frozen lambda_spa = {CALIB_LAMBDA}")


In [ ]:
# ---------- runner (skip_done + per-combo git push) ----------
REG = os.path.join(OUT, "registry.jsonl")

def done_set():
    s = set()
    if os.path.exists(REG):
        for line in open(REG):
            r = json.loads(line)
            s.add((r["combo"], r["dataset"], r["seed"]))
    return s

def run_combo(combo_name, cfg, data, seed):
    t0 = time.time()
    g = build_graph(data["coords"], DEVICE)
    k = data["k"]
    head = cfg.get("head", "kmeans")
    Z, r2r, r2a, mu = train_embed(cfg, data, g, seed, kl_head=(head == "idec"))
    km_labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(Z).labels_
    sil = float(silhouette_score(Z, km_labels))
    recbal = recon_balance(r2r, r2a)
    out_labels, extra = km_labels, {}
    if head == "idec" and mu is not None:
        q = student_q(torch.tensor(Z, dtype=torch.float32, device=DEVICE), mu)
        out_labels = q.argmax(1).cpu().numpy()
        extra = {"agree_ari_idec_vs_kmeans": float(adjusted_rand_score(km_labels, out_labels))}
    post_ari = float(adjusted_rand_score(data["gt"], out_labels))  # quarantined: reporting only
    d = os.path.join(OUT, combo_name, data["_ds"], f"seed{seed}")
    os.makedirs(d, exist_ok=True)
    np.save(os.path.join(d, "fused_embedding.npy"), Z)
    np.save(os.path.join(d, "labels.npy"), out_labels)
    metrics = {"silhouette": sil, "reconstruction_balance": recbal,
               "r2_rna": r2r, "r2_aux": r2a, "k": k,
               "wall_time_seconds": round(time.time() - t0, 2), **extra}
    json.dump(metrics, open(os.path.join(d, "metrics.json"), "w"), indent=2)
    json.dump({"ari": post_ari, "quarantined_reporting_only": True},
              open(os.path.join(d, "posthoc.json"), "w"), indent=2)
    with open(REG, "a") as f:
        f.write(json.dumps({"combo": combo_name, "dataset": data["_ds"], "seed": seed,
                            "metrics": metrics, "posthoc_ari": post_ari, "cfg": cfg}) + "\n")
    print(f"done {combo_name}/{data['_ds']}/seed{seed} sil={sil:.3f} recbal={recbal:.3f} "
          f"R2r={r2r:.3f} posthocARI={post_ari:.3f} [{time.time()-t0:.1f}s]", flush=True)

for combo_name, cfg in COMBINATIONS:
    for ds_id in DATASETS:
        for seed in SEEDS:
            if (combo_name, ds_id, seed) in done_set():
                print(f"skip done {combo_name}/{ds_id}/{seed}", flush=True)
                continue
            run_combo(combo_name, cfg, DATA_CACHE[ds_id], seed)
    git_push([f"runs/s4/{combo_name}", "runs/s4/registry.jsonl"], f"s4: {combo_name}")

print("All Stage 4 runs complete!")


In [ ]:
# ---------- RAUS v3b Ranking & Aggregate Summary ----------
from collections import defaultdict

def seed_stability(labs):
    vals = []
    ss = sorted(labs)
    for i in range(len(ss)):
        for j in range(i + 1, len(ss)):
            vals.append(adjusted_rand_score(labs[ss[i]], labs[ss[j]]))
    return float(np.mean(vals))

per_ds = defaultdict(list)
combos = [c for c, _ in COMBINATIONS]
for combo in combos:
    for ds_id in DATASETS:
        sils, stab_labs, recbals, r2rs, paris = [], {}, [], [], []
        ok = True
        for seed in SEEDS:
            d = os.path.join(OUT, combo, ds_id, f"seed{seed}")
            mp, pp = os.path.join(d, "metrics.json"), os.path.join(d, "posthoc.json")
            if not (os.path.exists(mp) and os.path.exists(pp)):
                ok = False; break
            m = json.load(open(mp)); p = json.load(open(pp))
            sils.append(m["silhouette"]); recbals.append(m["reconstruction_balance"])
            r2rs.append(m.get("r2_rna", 0.0))
            paris.append(p["ari"]); stab_labs[seed] = np.load(os.path.join(d, "labels.npy"))
        if not ok:
            continue
        per_ds[ds_id].append({
            "combo": combo,
            "sil": float(np.mean(sils)),
            "stab": seed_stability(stab_labs),
            "recbal": float(np.mean(recbals)),
            "r2_rna": float(np.mean(r2rs)),
            "pari": float(np.mean(paris)),
        })

def borda_v3b(rows):
    # RAUS v3b: Borda{Stability, RecBal, R2_rna} (drop silhouette)
    pts = {r["combo"]: 0 for r in rows}
    for m in ["stab", "recbal", "r2_rna"]:
        srt = sorted(rows, key=lambda r: r[m], reverse=True)
        for i, r in enumerate(srt):
            pts[r["combo"]] += i + 1
    return pts

cross = defaultdict(list)
for ds_id in sorted(per_ds):
    rows = per_ds[ds_id]
    pts = borda_v3b(rows)
    by = {r["combo"]: r for r in rows}
    print(f"\n=== {ds_id} (RAUS v3b: Stability + RecBal + R2_rna; lower better) ===")
    for rank_idx, combo in enumerate(sorted(pts, key=pts.get), 1):
        r = by[combo]
        print(f"#{rank_idx:<2} {combo:16} borda={pts[combo]:2d} stab={r['stab']:.3f} "
              f"recbal={r['recbal']:.3f} R2r={r['r2_rna']:.3f} sil={r['sil']:.3f} postARI={r['pari']:.3f}")
        cross[combo].append((ds_id, pts[combo]))

print("\n=== Cross-Dataset Aggregate (RAUS v3b Borda) ===")
for combo in sorted(cross, key=lambda c: sum(r for _, r in cross[c])):
    rs = cross[combo]
    avg = sum(r for _, r in rs) / len(rs)
    detail = " ".join(f"{d}#{r}" for d, r in rs)
    print(f"{combo:16} avg_borda={avg:5.2f}  {detail}")

json.dump({c: [{"dataset": d, "borda_v3b": b} for d, b in v] for c, v in cross.items()},
          open(os.path.join(OUT, "summary.json"), "w"), indent=2)
git_push(["runs/s4/summary.json", "runs/s4/registry.jsonl"], "s4: summary + registry")
print("Stage 4 complete & saved under", OUT)
